In [2]:
import requests
import pandas as pd

url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
params = {
    "make": "HONDA",
    "model": "CIVIC",
    "modelYear": 2018
}

response = requests.get(url, params=params)
data = response.json()

print(len(data['results']))

598


In [3]:
df = pd.DataFrame(data["results"])

print(df.columns)
print(df.head())

Index(['odiNumber', 'manufacturer', 'crash', 'fire', 'numberOfInjuries',
       'numberOfDeaths', 'dateOfIncident', 'dateComplaintFiled', 'vin',
       'components', 'summary', 'products'],
      dtype='object')
   odiNumber                      manufacturer  crash   fire  \
0   11678465  Honda (American Honda Motor Co.)  False  False   
1   11677761  Honda (American Honda Motor Co.)  False  False   
2   11676074  Honda (American Honda Motor Co.)  False  False   
3   11675834  Honda (American Honda Motor Co.)  False  False   
4   11675678  Honda (American Honda Motor Co.)  False  False   

   numberOfInjuries  numberOfDeaths dateOfIncident dateComplaintFiled  \
0                 0               0     07/28/2025         08/05/2025   
1                 0               0     08/01/2025         08/01/2025   
2                 0               0     07/08/2025         07/25/2025   
3                 0               0     06/01/2023         07/24/2025   
4                 0               0   

In [4]:
components = (
    df["components"]
    .str.split(",")
    .explode()
    .str.strip()
)

ranking = components.value_counts()

print(ranking.head(10))

components
STEERING                  224
UNKNOWN OR OTHER          105
GASOLINE                   89
FUEL SYSTEM                89
FUEL/PROPULSION SYSTEM     70
ELECTRICAL SYSTEM          51
ENGINE                     50
SERVICE BRAKES             21
AIR BAGS                   20
STRUCTURE                  19
Name: count, dtype: int64


In [11]:
ranking = components.value_counts()

ranking_df = ranking.reset_index()
ranking_df.columns = ["component", "count"]

ranking_df["percentage"] = (
    ranking_df["count"] / len(df) * 100
).round(1)

print(ranking_df.head(10))

                component  count  percentage
0                STEERING    224        37.5
1        UNKNOWN OR OTHER    105        17.6
2                GASOLINE     89        14.9
3             FUEL SYSTEM     89        14.9
4  FUEL/PROPULSION SYSTEM     70        11.7
5       ELECTRICAL SYSTEM     51         8.5
6                  ENGINE     50         8.4
7          SERVICE BRAKES     21         3.5
8                AIR BAGS     20         3.3
9               STRUCTURE     19         3.2


In [12]:
print(df[[
    "components",
    "summary",
    "crash",
    "fire",
    "numberOfInjuries",
    "numberOfDeaths"
]].head())

                   components  \
0  STEERING,ELECTRICAL SYSTEM   
1                    STEERING   
2                    STEERING   
3                    STEERING   
4                    STEERING   

                                             summary  crash   fire  \
0  The electric power steering (EPS) system faile...  False  False   
1  The contact owns a 2018 Honda Civic. The conta...  False  False   
2  We have noticed a clicking noise and her steer...  False  False   
3         At speed the steering sticks when turning.  False  False   
4  The contact owns a 2018 Honda Civic. The conta...  False  False   

   numberOfInjuries  numberOfDeaths  
0                 0               0  
1                 0               0  
2                 0               0  
3                 0               0  
4                 0               0  


In [13]:
ranking_df_filtered = ranking_df[
    ranking_df["component"] != "UNKNOWN OR OTHER"
]

print(ranking_df_filtered.head(10))

                 component  count  percentage
0                 STEERING    224        37.5
2                 GASOLINE     89        14.9
3              FUEL SYSTEM     89        14.9
4   FUEL/PROPULSION SYSTEM     70        11.7
5        ELECTRICAL SYSTEM     51         8.5
6                   ENGINE     50         8.4
7           SERVICE BRAKES     21         3.5
8                 AIR BAGS     20         3.3
9                STRUCTURE     19         3.2
10             POWER TRAIN     14         2.3


In [14]:
def get_component_ranking(df):
    components = (
        df["components"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
    )

    ranking = (
        components
        .value_counts()
        .rename_axis("component")
        .reset_index(name="count")
    )

    ranking["percentage"] = (
        ranking["count"] / len(df) * 100
    ).round(1)

    return ranking

ranking = get_component_ranking(df)
print(ranking.head(10))

                component  count  percentage
0                STEERING    224        37.5
1        UNKNOWN OR OTHER    105        17.6
2                GASOLINE     89        14.9
3             FUEL SYSTEM     89        14.9
4  FUEL/PROPULSION SYSTEM     70        11.7
5       ELECTRICAL SYSTEM     51         8.5
6                  ENGINE     50         8.4
7          SERVICE BRAKES     21         3.5
8                AIR BAGS     20         3.3
9               STRUCTURE     19         3.2
